# Pharmaceutical Sales Time-Series Forecasting with ARIMA

## Student Solution Notebook

**Dataset:** `novartis_time_series_pharma_sales_600_rows.csv`

> **Important:** The dataset is synthetic and created only for educational purposes. It is not Novartis proprietary data and must not be used for real commercial, clinical, regulatory, supply, or patient decisions.

### Learning objectives
- Understand a time series
- Analyze trend and seasonality
- Test stationarity
- Apply differencing
- Understand ACF/PACF
- Build ARIMA models
- Validate chronologically
- Compare forecasts using MAE/RMSE
- Diagnose residuals
- Produce a 30-day future forecast


## 1. Business Problem

A pharmaceutical commercial analytics team wants to forecast daily product demand.

The target is:

**`Sales_Units`**

We will use historical daily sales to forecast future sales and answer:

> **How many units should we expect to sell over the next forecasting period?**

We will use ARIMA as the main statistical forecasting model.


In [ ]:
# Install packages if needed
# !pip install pandas numpy matplotlib seaborn statsmodels scikit-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the Dataset

We first load the CSV, inspect its dimensions, and understand the columns.


In [ ]:
# Change this path if your CSV is stored elsewhere
file_path = "novartis_time_series_pharma_sales_600_rows.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()


In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Prepare the Time Series

ARIMA needs observations to be in chronological order.

We convert `Date` to datetime, sort by date, and use it as the index.


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df = df.set_index("Date")

sales = df["Sales_Units"].copy()

print("Start:", sales.index.min())
print("End:", sales.index.max())
print("Observations:", len(sales))


## 4. Exploratory Data Analysis

The first question is:

> What does the sales series look like?

Look for:
- upward/downward trend
- repeated patterns
- unusual observations
- changing variability


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(sales)
plt.title("Daily Pharmaceutical Sales")
plt.xlabel("Date")
plt.ylabel("Sales Units")
plt.show()


In [ ]:
print(sales.describe())


### Weekly pattern

Daily pharmaceutical sales may behave differently on weekdays and weekends.

We calculate average sales by day of week.


In [ ]:
dow_summary = df.groupby("Day_of_Week")["Sales_Units"].mean()

# Put days in calendar order
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

dow_summary = dow_summary.reindex(day_order)
dow_summary


In [ ]:
plt.figure(figsize=(10, 4))
dow_summary.plot(kind="bar")
plt.title("Average Sales by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Sales Units")
plt.xticks(rotation=45)
plt.show()


## 5. Promotion and Sales

The dataset contains an external variable called `Promotion`.

This is useful for business analysis, but **standard ARIMA will primarily model the sales series itself**. Later we discuss SARIMAX/ARIMAX for external predictors.


In [ ]:
promotion_summary = df.groupby("Promotion")["Sales_Units"].agg(
    ["mean", "median", "count"]
)

promotion_summary


## 6. Stationarity

ARIMA works particularly well when the modeled series is stationary.

A stationary series has relatively stable statistical properties over time.

We use the **Augmented Dickey-Fuller (ADF)** test.

### Hypotheses

- H0: The series has a unit root / is non-stationary.
- H1: The series is stationary.

A small p-value, commonly below 0.05, provides evidence against H0.


In [ ]:
def adf_test(series, name="Series"):
    result = adfuller(series.dropna())

    print(f"ADF Test: {name}")
    print("-" * 40)
    print("ADF Statistic:", result[0])
    print("p-value:", result[1])
    print("Used lags:", result[2])
    print("Number of observations:", result[3])

    print("\nCritical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value}")

    if result[1] < 0.05:
        print("\nConclusion: Reject H0 → evidence of stationarity.")
    else:
        print("\nConclusion: Fail to reject H0 → series may be non-stationary.")

adf_test(sales, "Original Sales")


## 7. First-Order Differencing

If the original series is non-stationary, we can difference it:

**Differenced value = Current value − Previous value**

Mathematically:

`Y'_t = Y_t - Y_(t-1)`

This corresponds to `d = 1` in ARIMA.


In [ ]:
sales_diff = sales.diff().dropna()

plt.figure(figsize=(14, 4))
plt.plot(sales_diff)
plt.title("First-Differenced Sales")
plt.xlabel("Date")
plt.ylabel("Differenced Sales")
plt.show()


In [ ]:
adf_test(sales_diff, "First-Differenced Sales")


## 8. ACF and PACF

ACF and PACF help us understand lag relationships and suggest candidate ARIMA parameters.

- **ACF** → correlation with previous observations
- **PACF** → direct correlation after accounting for intermediate lags

We use them as guidance, not as the only method for selecting the final model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(sales_diff, lags=30, ax=axes[0])
axes[0].set_title("ACF - Differenced Sales")

plot_pacf(sales_diff, lags=30, ax=axes[1], method="ywm")
axes[1].set_title("PACF - Differenced Sales")

plt.tight_layout()
plt.show()


## 9. Train/Test Split

Time series must be split chronologically.

We will use:

- First 80% → training
- Last 20% → testing

### Important

Do **not** randomly shuffle time-series observations.

The model must never train on future information.


In [ ]:
split_point = int(len(sales) * 0.80)

train = sales.iloc[:split_point]
test = sales.iloc[split_point:]

print("Train observations:", len(train))
print("Test observations:", len(test))
print("Train end:", train.index.max())
print("Test start:", test.index.min())


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train, label="Train")
plt.plot(test, label="Test")
plt.axvline(test.index[0], linestyle="--", label="Train/Test Boundary")

plt.title("Chronological Train/Test Split")
plt.xlabel("Date")
plt.ylabel("Sales Units")
plt.legend()
plt.show()


## 10. Naive Baseline

Before using ARIMA, establish a simple baseline.

The naive forecast assumes:

> Tomorrow's sales = today's sales.

This is important because an advanced model is only useful if it beats a simple baseline.


In [ ]:
naive_forecast = pd.Series(
    train.iloc[-1],
    index=test.index
)

naive_mae = mean_absolute_error(test, naive_forecast)
naive_rmse = np.sqrt(mean_squared_error(test, naive_forecast))

print("Naive MAE :", naive_mae)
print("Naive RMSE:", naive_rmse)


## 11. Build Candidate ARIMA Models

ARIMA is written as:

`ARIMA(p, d, q)`

Where:

- `p` = number of autoregressive lags
- `d` = number of differences
- `q` = number of moving-average error terms

We will test several reasonable candidate models.


In [ ]:
candidate_orders = [
    (1, 1, 0),
    (1, 1, 1),
    (2, 1, 0),
    (2, 1, 1),
    (3, 1, 0),
    (3, 1, 1)
]

results = []

for order in candidate_orders:
    try:
        model = ARIMA(train, order=order)
        fitted = model.fit()

        forecast = fitted.forecast(steps=len(test))

        mae = mean_absolute_error(test, forecast)
        rmse = np.sqrt(mean_squared_error(test, forecast))

        results.append({
            "Model": f"ARIMA{order}",
            "Order": order,
            "MAE": mae,
            "RMSE": rmse,
            "AIC": fitted.aic
        })

    except Exception as e:
        print(f"Failed for {order}: {e}")

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


## 12. Select the Best ARIMA Model

We select the model using **out-of-sample test performance**.

Lower MAE and RMSE are better.

AIC can also be useful for model comparison, but forecasting performance on unseen data is especially important for this exercise.


In [ ]:
best_order = results_df.iloc[0]["Order"]

print("Best ARIMA order based on RMSE:", best_order)
print(results_df.iloc[0])


## 13. Evaluate the Best Model Visually

Now fit the selected ARIMA model on the training data and compare its test forecasts against actual sales.


In [ ]:
best_model = ARIMA(train, order=best_order)
best_model_fit = best_model.fit()

arima_forecast = best_model_fit.forecast(steps=len(test))

arima_mae = mean_absolute_error(test, arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test, arima_forecast))

print("Best model:", best_order)
print("MAE :", arima_mae)
print("RMSE:", arima_rmse)


In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(train, label="Training Data")
plt.plot(test, label="Actual Test Data")
plt.plot(arima_forecast, label=f"ARIMA{best_order} Forecast")

plt.title("ARIMA Forecast vs Actual Sales")
plt.xlabel("Date")
plt.ylabel("Sales Units")
plt.legend()
plt.show()


## 14. Compare Naive vs ARIMA

A model should demonstrate improvement over a simple baseline.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Naive", f"ARIMA{best_order}"],
    "MAE": [naive_mae, arima_mae],
    "RMSE": [naive_rmse, arima_rmse]
})

comparison


In [ ]:
plt.figure(figsize=(9, 4))
comparison.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar")

plt.title("Forecast Error Comparison")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.show()


## 15. Residual Diagnostics

Residual:

`Residual = Actual − Forecast`

A good forecasting model should leave residuals that contain little predictable structure.

We inspect:
- residual mean
- residual plot
- residual ACF


In [ ]:
residuals = test - arima_forecast

print("Residual mean:", residuals.mean())
print("\nResidual summary:")
print(residuals.describe())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(residuals)
axes[0].axhline(0, linestyle="--")
axes[0].set_title("ARIMA Forecast Residuals")
axes[0].set_ylabel("Residual")

plot_acf(residuals.dropna(), lags=30, ax=axes[1])
axes[1].set_title("Residual ACF")

plt.tight_layout()
plt.show()


## 16. Refit the Selected Model on All Historical Data

After evaluating the model on the test period, we can use all available historical data to train the final model.

Then we forecast the next 30 days.


In [ ]:
final_model = ARIMA(sales, order=best_order)
final_model_fit = final_model.fit()

future_forecast = final_model_fit.forecast(steps=30)

future_df = pd.DataFrame({
    "Date": future_forecast.index,
    "Forecast_Sales_Units": np.round(future_forecast.values, 0).astype(int)
})

future_df.head(10)


In [ ]:
plt.figure(figsize=(14, 5))

# Show recent history
recent_history = sales.tail(120)

plt.plot(
    recent_history.index,
    recent_history.values,
    label="Historical Sales"
)

plt.plot(
    future_forecast.index,
    future_forecast.values,
    label="30-Day Forecast"
)

plt.axvline(
    sales.index[-1],
    linestyle="--",
    label="Forecast Start"
)

plt.title(f"30-Day Future Sales Forecast - ARIMA{best_order}")
plt.xlabel("Date")
plt.ylabel("Sales Units")
plt.legend()
plt.show()


## 17. Future Forecast Table

The following table contains the predicted sales for the next 30 days.


In [ ]:
future_df


## 18. Business Interpretation

### How to interpret MAE

If MAE = 100:

> The model is wrong by approximately 100 sales units per day on average.

### How to interpret RMSE

RMSE penalizes large errors more strongly than MAE.

Therefore, if RMSE is much larger than MAE, there may be some larger forecast errors or unusual observations.

### Important limitation

Standard ARIMA mainly models the historical behavior of `Sales_Units`.

Variables such as:
- promotion
- marketing spend
- price
- inventory
- region

may also influence demand.

To explicitly incorporate external variables, consider:

**SARIMAX / ARIMAX**

or feature-based machine-learning models.


## 19. Student Questions

Answer these in your own words:

1. Does the original sales series appear stationary?
2. Why did we apply differencing?
3. What do `p`, `d`, and `q` mean?
4. Which ARIMA model performed best?
5. Did ARIMA beat the naive baseline?
6. What do MAE and RMSE tell the business?
7. Do the residuals show remaining patterns?
8. Why might promotion and marketing spend matter?
9. Why would SARIMA potentially be useful for daily sales?
10. What are the limitations of this forecasting exercise?


# 20. Advanced Challenge — SARIMA

The EDA may reveal weekly seasonality in daily data.

A natural extension is **SARIMA**, which adds seasonal AR, differencing, and moving-average components.

General form:

`SARIMA(p,d,q)(P,D,Q,s)`

For daily data with weekly seasonality:

`s = 7`

Example candidate:

`SARIMA(1,1,1)(1,1,1,7)`

Try this only after completing the ARIMA workflow.

Then compare:

- Naive
- ARIMA
- SARIMA

using the same chronological test set.


# Key Takeaways

Remember:

**ARIMA(p,d,q)**

- `p` → past values
- `d` → differencing
- `q` → past forecast errors

The professional forecasting workflow is:

**Understand → Visualize → Check stationarity → Difference → ACF/PACF → Train chronologically → Forecast → Evaluate → Diagnose → Refit → Forecast future**

> Never randomly shuffle a time series when building a forecasting model.
